# AITravelPlanner：五智能体协作的 AI 旅行规划助手

本 Notebook 演示 HelloAgents 的教学版实现：**需求解析 → 目的地调研 → 行程规划 → 预算测算 → 方案校验**。

```mermaid
flowchart LR
    U["自然语言需求"] --> A1["需求解析"] --> A2["调研(工具)"] --> A3["行程规划"] --> A4["预算(工具)"] --> A5{"校验"}
    A5 -->|"< 7 分"| A3
    A5 -->|"通过"| OUT["Markdown 报告"]
```

运行前请先复制 `.env.example` 为 `.env` 并填入模型配置。

In [ ]:
# 读取 .env 并检查配置是否齐全（不打印密钥本身）
import os
from dotenv import load_dotenv

load_dotenv()

required = ["LLM_MODEL_ID", "LLM_API_KEY", "LLM_BASE_URL"]
missing = [k for k in required if not os.getenv(k)]
print("配置完整 ✅" if not missing else f"缺少环境变量：{missing}")

## 1. 先看工具（不消耗 Token）

工具是智能体的「手」：景点检索读本地知识库，天气是确定性演示数据，预算做四则运算。
先单独调用，确认数据源正常，再交给智能体。

In [ ]:
from src.tools import build_tools

tools = {t.name: t for t in build_tools()}

print(tools["search_attractions"].run({"city": "杭州", "keyword": "博物馆"}))
print(tools["get_weather"].run({"city": "杭州", "date": "2026-12-01"}))
print(tools["calc_budget"].run({
    "days": 3, "people": 2, "transport": 500,
    "hotel_per_night": 400, "food_per_day": 150, "tickets": 150,
}))

## 2. 运行完整流水线

下面的调用会真实请求大模型：需求解析 → 调研 → 规划 → 预算 → 校验（分数低会自动返工一次）。

In [ ]:
from src.pipeline import run_pipeline

result = run_pipeline("杭州 3 天 2 人，预算 3000 元，喜欢美食和博物馆")
print(result["report"])

## 3. 查看结构化中间结果

报告背后的每一步都可以单独检查，方便定位是解析错了、还是规划不合理。

In [ ]:
import json

for key in ["requirement", "budget", "review"]:
    print(f"===== {key} =====")
    print(json.dumps(result[key], ensure_ascii=False, indent=2))

## 4. 换一个需求试试

把下面的字符串改成你自己的需求，例如：`成都 2 天 1 人，预算 1500 元，喜欢美食和大熊猫`。

In [ ]:
my_input = "成都 2 天 1 人，预算 1500 元，喜欢美食"
my_result = run_pipeline(my_input)
print(my_result["report"])